# PhishGuard — exploring signals and errors

Use this notebook to inspect which signals fire on a dataset split, find false positives, and sanity-check weights before retraining.

**Privacy:** processed splits contain real phishing emails and may contain PII (Enron). Do not commit outputs; clear them before saving.

In [ ]:
import sys, json
from pathlib import Path
from collections import Counter
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from phishguard.analyzer import Analyzer
from phishguard.config import Settings

an = Analyzer(Settings(), providers=[])
split = ROOT / 'data' / 'processed' / 'val.jsonl'
src = split if split.exists() else ROOT / 'data' / 'seed' / 'samples.jsonl'
rows = [json.loads(l) for l in src.read_text(encoding='utf-8').splitlines() if l.strip()]
rows = [{'content': r.get('content', r.get('text')), 'input_type': r.get('input_type', 'text'), 'label': r['label']} for r in rows]
len(rows)

In [ ]:
results = [(r, an.analyze(r['content'], r['input_type'])) for r in rows]
fires = {0: Counter(), 1: Counter()}
for r, res in results:
    fires[r['label']].update(f.id for f in res.red_flags)
print('Signals on LEGIT samples (false-positive pressure):', fires[0].most_common(10))
print('Signals on MALICIOUS samples:', fires[1].most_common(10))

In [ ]:
# Worst errors: legit with high risk, malicious with low risk
fps = sorted([(res.risk_score, r['content'][:90]) for r, res in results if r['label'] == 0], reverse=True)[:5]
fns = sorted([(res.risk_score, r['content'][:90]) for r, res in results if r['label'] == 1])[:5]
print('Top false-positive candidates:'); [print(' ', s, t) for s, t in fps]
print('Top false-negative candidates:'); [print(' ', s, t) for s, t in fns]